# Intro

This notebook gives an intro to our project and its components. It also showcases what can be done

## Load & Inspect OCELs

First, we can load ocels and inspect them

In [6]:
import pm4py

ocel = pm4py.read_ocel2_json("../data/ocel2-p2p.json")

In [7]:
ocel.events

,ocel:eid,ocel:timestamp,ocel:activity,lifecycle,resource
9786,event:1,2022-04-01 09:26:00+00:00,Create Purchase Requisition,complete,Manufacturing Department
1598,event:3,2022-04-02 14:16:00+00:00,Approve Purchase Requisition,complete,Procurement Requisition Manager
9787,event:9,2022-04-04 08:37:00+00:00,Create Purchase Requisition,complete,Manufacturing Department
9788,event:11,2022-04-04 09:25:00+00:00,Create Purchase Requisition,complete,Manufacturing Department
11442,event:13,2022-04-04 09:44:00+00:00,Delegate Purchase Requisition Approval,complete,Manufacturing Department
...,...,...,...,...,...
8187,event:29334,2024-10-29 07:55:00+00:00,Create Invoice Receipt,complete,Finance/Account Department
14670,event:29336,2024-10-29 08:10:00+00:00,Perform Two-Way Match,complete,Finance/Account Department
12727,event:29338,2024-10-29 14:14:00+00:00,Execute Payment,complete,Finance/Account Department
12728,event:29339,2024-10-29 22:22:00+00:00,Execute Payment,complete,Finance/Account Department


In [10]:
ocel_new = pm4py.read_ocel2_sqlite("../data/new.sqlite")

In [3]:
import sqlite3
import polars as pl

conn = sqlite3.connect("../data/new.sqlite")
objects = pl.read_database("SELECT * FROM object", conn)
events = pl.read_database("SELECT * FROM event", conn)
conn.close()
print(objects)
events

shape: (4, 2)
┌─────────┬───────────┐
│ ocel_id ┆ ocel_type │
│ ---     ┆ ---       │
│ str     ┆ str       │
╞═════════╪═══════════╡
│ user1   ┆ user      │
│ user2   ┆ user      │
│ order1  ┆ order     │
│ user1   ┆ user      │
└─────────┴───────────┘


ocel_id,ocel_type
str,str
"""event1""","""Create Order"""
"""event2""","""Receive Order"""


# Issues detection

## N6 - Incorrect Object
(a) complete duplicate including IDs

(b) same content but different IDs

In [33]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/new.sqlite")
objects = pl.read_database("SELECT * FROM object", conn)

# (a) totally duplicated: same ocel_id and ocel_type
dupes_a = objects[objects.duplicated(keep=False)]
print("(a) Totally duplicated objects:")
print(dupes_a if not dupes_a.empty else "  None found")

# (b) same content but different IDs: check type-specific tables directly
print("\n(b) Same attributes, different IDs:")
types = objects["ocel_type"].unique()
for t in types:
    try:
        attrs = pd.read_sql(f"SELECT * FROM object_{t}", conn)
        attr_cols = [c for c in attrs.columns if c != "ocel_id"]
        dupes_b = attrs[attrs.duplicated(subset=attr_cols, keep=False)]
        if not dupes_b.empty:
            print(f"  Type '{t}':")
            print(dupes_b)
        else:
            print(f"  Type '{t}': None found")
    except Exception:
        print(f"  Type '{t}': no attribute table")

conn.close()

(a) Totally duplicated objects:
  ocel_id ocel_type
0   user1      user
3   user1      user

(b) Same attributes, different IDs:
  Type 'user':
  ocel_id        name  purchases          address
1   user2  Jane Smith         10  Johnson Road 56
2   user3  Jane Smith         10  Johnson Road 56
  Type 'order': None found
  Type '     ': no attribute table


### Discussion point:

Other type of duplicates: 
- duplicate meaning but different IDs and some (not important) attributes

False positive:
- same attributes not always mean the duplicated (ex. name, address could be not a duplicate). Needed at least data knowledge for attribute set, that we consider as a duplicate.

## N2 - Missing Object Type

An object has a missing or unknown type (`ocel_type` is NULL or empty)

In [31]:
conn = sqlite3.connect("../data/new.sqlite")
objects = pl.read_database("SELECT * FROM object", conn)
conn.close()

missing_type = objects[objects["ocel_type"].isna() | (objects["ocel_type"].str.strip() == "")]
print("(N2) Objects with missing type:")
print(missing_type if not missing_type.empty else "  None found")

(N2) Objects with missing type:
  ocel_id ocel_type
4   user5          


## N10 - Incorrect E2O Relation

An E2O relation is erroneously logged: 
- (a) relation with existing object → data knowledge needed
- (b) relation with non-existing object: Detected by finding E2O entries whose `ocel_event_id` or `ocel_object_id` has no matching record in the `event` or `object` table.

In [32]:
import polars as pl

conn = sqlite3.connect("../data/new.sqlite")
e2o = pl.read_database("SELECT * FROM event_object", conn)
events = pl.read_database("SELECT ocel_id FROM event", conn)
objects = pl.read_database("SELECT ocel_id FROM object", conn).unique()
conn.close()

invalid_event = e2o.join(events, left_on="ocel_event_id", right_on="ocel_id", how="anti")
invalid_object = e2o.join(objects, left_on="ocel_object_id", right_on="ocel_id", how="anti")

print("(N10) E2O relations with non-existent event:")
print(invalid_event if not invalid_event.is_empty() else "  None found")

print("\n(N10) E2O relations with non-existent object:")
print(invalid_object if not invalid_object.is_empty() else "  None found")

(N10) E2O relations with non-existent event:
  None found

(N10) E2O relations with non-existent object:
shape: (1, 3)
┌────────────────┬───────────────┬────────────────┐
│ ocel_object_id ┆ ocel_event_id ┆ ocel_qualifier │
│ ---            ┆ ---           ┆ ---            │
│ str            ┆ str           ┆ str            │
╞════════════════╪═══════════════╪════════════════╡
│ order4         ┆ event1        ┆ event of order │
└────────────────┴───────────────┴────────────────┘
